<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/DayChallenge_W8_D4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# Création de l'arborescence
os.makedirs('output', exist_ok=True)

# 1. requirements.txt
with open('requirements.txt', 'w') as f:
    f.write('fastapi\nuvicorn\nhttpx\nbeautifulsoup4\nreadability-lxml\npydantic\npython-dotenv\n')

# 2. .env.example
with open('.env.example', 'w') as f:
    f.write('MCP_HTTP_TOKEN=votre_token_secret\nTAVILY_API_KEY=votre_cle_tavily\nOLLAMA_BASE_URL=http://localhost:11434\n')

print('Fichiers de base créés.')

Fichiers de base créés.


In [2]:
%%writefile config.py
import os
from dotenv import load_dotenv

load_dotenv()

MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "default_secret")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OUTPUT_DIR = "output"

Writing config.py


In [3]:
%%writefile models.py
from pydantic import BaseModel, HttpUrl, Field
from typing import List, Optional

class SearchRequest(BaseModel):
    query: str = Field(..., min_length=1)
    k: int = Field(default=5, gt=0)

class SearchResult(BaseModel):
    title: str
    url: str
    snippet: str
    source: str

class FetchRequest(BaseModel):
    url: HttpUrl

class FetchResponse(BaseModel):
    url: str
    title: str
    text: str

class SummarizeRequest(BaseModel):
    topic: str = Field(..., min_length=1)
    docs: List[SearchResult]

class Source(BaseModel):
    i: int
    title: str
    url: str

class SummarizeResponse(BaseModel):
    bullets: List[str]
    sources: List[Source]

class SaveRequest(BaseModel):
    filename: str
    content: str

class SaveResponse(BaseModel):
    path: str

Writing models.py


In [4]:
%%writefile search.py
import httpx
from typing import List
from config import TAVILY_API_KEY
from models import SearchResult

async def search_tavily(query: str, k: int = 5) -> List[SearchResult]:
    url = "https://api.tavily.com/search"
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "search_depth": "basic",
        "max_results": k
    }
    async with httpx.AsyncClient(timeout=10.0) as client:
        response = await client.post(url, json=payload)
        response.raise_for_status()
        data = response.json()

        results = []
        for item in data.get("results", []):
            results.append(SearchResult(
                title=item.get("title", "No Title"),
                url=item.get("url", ""),
                snippet=item.get("content", ""),
                source=item.get("url", "")
            ))
        return results

Writing search.py


In [5]:
%%writefile fetch.py
import httpx
from bs4 import BeautifulSoup
from readability import Document
from models import FetchResponse

async def fetch_readable(url: str) -> FetchResponse:
    async with httpx.AsyncClient(timeout=15.0, follow_redirects=True) as client:
        response = await client.get(url)
        response.raise_for_status()

        doc = Document(response.text)
        summary_html = doc.summary()
        soup = BeautifulSoup(summary_html, "lxml")
        clean_text = soup.get_text(separator=" ", strip=True)

        return FetchResponse(
            url=url,
            title=doc.title(),
            text=clean_text[:5000] # Limite pour éviter de saturer le LLM
        )

Writing fetch.py


In [6]:
%%writefile summarizer.py
import httpx
import json
from config import OLLAMA_BASE_URL
from models import SummarizeRequest, SummarizeResponse, Source

async def summarize_with_ollama(req: SummarizeRequest) -> SummarizeResponse:
    context = "\n".join([f"[{i+1}] Source: {d.title}\nContent: {d.snippet}" for i, d in enumerate(req.docs)])

    prompt = f"""Task: Summarize the topic '{req.topic}' based on the sources provided.
Guidelines:
1. Provide exactly 5 bullet points.
2. Each bullet point must be less than 200 characters.
3. Include citations like [1], [2] based on the sources.
4. Output ONLY a JSON object with a 'bullets' key containing the list of strings.

Sources:
{context}"""

    async with httpx.AsyncClient(timeout=60.0) as client:
        response = await client.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json={"model": "llama3", "prompt": prompt, "stream": false, "format": "json"}
        )
        response.raise_for_status()
        raw_text = response.json().get("response", "{}")
        data = json.loads(raw_text)

        bullets = data.get("bullets", [])[:5]
        sources = [Source(i=i+1, title=d.title, url=d.url) for i, d in enumerate(req.docs)]

        return SummarizeResponse(bullets=bullets, sources=sources)

Writing summarizer.py


In [7]:
%%writefile auth.py
from fastapi import Security, HTTPException, status
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from config import MCP_HTTP_TOKEN

security = HTTPBearer()

def validate_token(credentials: HTTPAuthorizationCredentials = Security(security)):
    if credentials.credentials != MCP_HTTP_TOKEN:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid or missing token",
            headers={"WWW-Authenticate": "Bearer"},
        )
    return credentials.credentials

Writing auth.py


In [8]:
%%writefile server.py
import os
from fastapi import FastAPI, Depends, HTTPException
from auth import validate_token
from models import *
from search import search_tavily
from fetch import fetch_readable as fetch_logic
from summarizer import summarize_with_ollama
from config import OUTPUT_DIR

app = FastAPI(title="Web Research Summary Bot")

@app.get("/tools", dependencies=[Depends(validate_token)])
def get_tools():
    return [
        {"name": "search_web", "description": "Search the web for a query", "input_schema": SearchRequest.schema()},
        {"name": "fetch_readable", "description": "Extract main content from a URL", "input_schema": FetchRequest.schema()},
        {"name": "summarize_with_citations", "description": "Generate a summary with citations", "input_schema": SummarizeRequest.schema()},
        {"name": "save_markdown", "description": "Save content to a markdown file", "input_schema": SaveRequest.schema()}
    ]

@app.post("/tools/search_web", response_model=List[SearchResult], dependencies=[Depends(validate_token)])
async def api_search(req: SearchRequest):
    try:
        return await search_tavily(req.query, req.k)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/tools/fetch_readable", response_model=FetchResponse, dependencies=[Depends(validate_token)])
async def api_fetch(req: FetchRequest):
    try:
        return await fetch_logic(str(req.url))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/tools/summarize_with_citations", response_model=SummarizeResponse, dependencies=[Depends(validate_token)])
async def api_summarize(req: SummarizeRequest):
    try:
        return await summarize_with_ollama(req)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/tools/save_markdown", response_model=SaveResponse, dependencies=[Depends(validate_token)])
async def api_save(req: SaveRequest):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    filepath = os.path.join(OUTPUT_DIR, req.filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(req.content)
    return SaveResponse(path=filepath)

Writing server.py


In [9]:
%%writefile client.py
import sys
import httpx
import asyncio
from datetime import datetime
from config import MCP_HTTP_TOKEN

BASE_URL = "http://localhost:8000"
HEADERS = {"Authorization": f"Bearer {MCP_HTTP_TOKEN}"}

async def run_pipeline(topic: str):
    async with httpx.AsyncClient(timeout=120.0) as client:
        print(f"[*] Searching for: {topic}...")
        r_search = await client.post(f"{BASE_URL}/tools/search_web", json={"query": topic, "k": 3}, headers=HEADERS)
        docs = r_search.json()

        print("[*] Fetching content and summarizing...")
        r_sum = await client.post(f"{BASE_URL}/tools/summarize_with_citations",
                                  json={"topic": topic, "docs": docs}, headers=HEADERS)
        summary = r_sum.json()

        # Building Markdown
        md = f"# {topic}\n\n## Summary\n\n"
        for bullet in summary['bullets']:
            md += f"- {bullet}\n"

        md += "\n## Sources\n\n"
        for src in summary['sources']:
            md += f"[{src['i']}] {src['title']}\n{src['url']}\n\n"

        filename = f"brief_{datetime.now().strftime('%Y-%m-%d')}.md"
        r_save = await client.post(f"{BASE_URL}/tools/save_markdown",
                                   json={"filename": filename, "content": md}, headers=HEADERS)

        print(f"\nSaved:\n{r_save.json()['path']}")

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python client.py \"Topic\"")
    else:
        asyncio.run(run_pipeline(sys.argv[1]))

Writing client.py


# Web Research Summary Bot (HTTP)

Ce projet implémente un bot de recherche automatisé avec une architecture client-serveur.

## Installation

1.  **Prérequis** : Python 3.10+, Ollama (avec le modèle `llama3`).
2.  **Installer les dépendances** :
    ```bash
    pip install -r requirements.txt
    ```
3.  **Configuration** :
    - Copiez `.env.example` vers `.env`.
    - Remplissez votre `TAVILY_API_KEY` et `MCP_HTTP_TOKEN`.

## Lancement

1.  **Démarrer le serveur** :
    ```bash
    uvicorn server:app --reload --port 8000
    ```
2.  **Lancer une recherche via le client** :
    ```bash
    python client.py "Impact de l'IA sur la médecine"
    ```

## Endpoints API

- `GET /tools` : Liste les outils disponibles.
- `POST /tools/search_web` : Recherche Web via Tavily.
- `POST /tools/fetch_readable` : Extraction de texte propre.
- `POST /tools/summarize_with_citations` : Synthèse LLM (Ollama).
- `POST /tools/save_markdown` : Persistance sur disque.

## Exemples cURL

Remplacez `<TOKEN>` par la valeur de `MCP_HTTP_TOKEN`.

### 1. Liste des outils
```bash
curl -X GET http://localhost:8000/tools -H "Authorization: Bearer <TOKEN>"
```

### 2. Recherche Web
```bash
curl -X POST http://localhost:8000/tools/search_web \
     -H "Authorization: Bearer <TOKEN>" \
     -H "Content-Type: application/json" \
     -d '{"query": "Tesla FSD updates 2024", "k": 3}'
```

### 3. Synthèse avec citations
```bash
curl -X POST http://localhost:8000/tools/summarize_with_citations \
     -H "Authorization: Bearer <TOKEN>" \
     -H "Content-Type: application/json" \
     -d '{
          "topic": "AI",
          "docs": [{"title": "Source1", "url": "http://x.com", "snippet": "AI is growing", "source": "x.com"}]
         }'
```